# League of Legends 개인 매치 데이터 EDA

개인 LoL 매치 데이터를 활용해 승패와 관련된 주요 지표를 탐색합니다.

## 분석 목표
- 승리/패배 경기의 플레이 지표 차이 확인
- 승패와 주요 지표 간 상관관계 확인
- 데스 구간별 승률 분석
- 게임 시간대별 승률 분석
- 요일별 승률 및 카이제곱 검정
- 챔피언/포지션별 성과 비교

> 주의: 최종 경기 통계는 경기 결과의 영향을 함께 받기 때문에, 상관관계를 인과관계로 해석하지 않습니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


## 1. 데이터 불러오기


In [ ]:
df = pd.read_csv("../data/processed/my_lol_games.csv")

print("shape:", df.shape)
df.head()


## 2. 기본 데이터 확인


In [ ]:
df.info()


In [ ]:
print("결측치")
display(df.isnull().sum())

print("\n중복 match_id:", df["match_id"].duplicated().sum())


In [ ]:
df.describe(include="all").T


## 3. 날짜 전처리

Riot API의 `gameStartTimestamp`는 UTC 기준입니다.

현재 CSV의 `game_date`가 UTC 기준으로 저장된 경우 한국 시간으로 변환합니다.
이미 KST로 저장한 CSV라면 이 셀의 변환 부분은 생략하세요.


In [ ]:
df["game_date"] = pd.to_datetime(df["game_date"], utc=True).dt.tz_convert("Asia/Seoul")
df[["game_date"]].head()


## 4. 전체 승률


In [ ]:
total_games = len(df)
wins = int(df["win"].sum())
losses = total_games - wins
win_rate = df["win"].mean() * 100

print(f"전체 경기 수: {total_games}")
print(f"승리: {wins}")
print(f"패배: {losses}")
print(f"승률: {win_rate:.2f}%")


## 5. 승리 / 패배 경기 지표 비교


In [ ]:
metrics = [
    "kills", "deaths", "assists", "kda",
    "cs_per_min", "gold_per_min", "damage_per_min", "vision_score"
]

comparison = pd.DataFrame({
    "패배 평균": df[df["win"] == 0][metrics].mean(),
    "승리 평균": df[df["win"] == 1][metrics].mean()
})

comparison["차이"] = comparison["승리 평균"] - comparison["패배 평균"]
comparison["차이율(%)"] = comparison["차이"] / comparison["패배 평균"] * 100
comparison.round(2)


In [ ]:
comparison_plot = comparison["차이율(%)"].sort_values()

ax = comparison_plot.plot(kind="barh", figsize=(9, 6))
ax.set_title("Difference in Player Metrics: Win vs Loss")
ax.set_xlabel("Difference (%)")
ax.set_ylabel("")
plt.axvline(0, linewidth=1)
plt.tight_layout()
plt.show()


## 6. 승패와 주요 지표의 상관관계


In [ ]:
corr_cols = [
    "win", "kills", "deaths", "assists", "kda",
    "cs_per_min", "gold_per_min", "damage_per_min", "vision_score"
]

win_corr = (
    df[corr_cols]
    .corr(numeric_only=True)["win"]
    .drop("win")
    .sort_values()
)

win_corr


In [ ]:
ax = win_corr.plot(kind="barh", figsize=(9, 6))
ax.set_title("Correlation with Win")
ax.set_xlabel("Correlation Coefficient")
ax.set_ylabel("")
plt.axvline(0, linewidth=1)
plt.tight_layout()
plt.show()


### 해석 주의
승리한 경기에서는 자연스럽게 KDA, 골드, 어시스트 등이 높아질 수 있습니다.  
따라서 위 결과는 **승패와 함께 나타난 관계**로 해석하고, 승리의 직접적인 원인이라고 단정하지 않습니다.


## 7. 데스 횟수 구간별 승률


In [ ]:
death_bins = [-1, 3, 5, 7, 9, float("inf")]
death_labels = ["0~3", "4~5", "6~7", "8~9", "10+"]

df["death_group"] = pd.cut(df["deaths"], bins=death_bins, labels=death_labels)

death_stats = (
    df.groupby("death_group", observed=False)
    .agg(games=("win", "count"), wins=("win", "sum"), win_rate=("win", "mean"))
)

death_stats["losses"] = death_stats["games"] - death_stats["wins"]
death_stats["win_rate"] *= 100
death_stats.round(2)


In [ ]:
ax = death_stats["win_rate"].plot(kind="bar", figsize=(8, 5))
ax.set_title("Win Rate by Number of Deaths")
ax.set_xlabel("Deaths")
ax.set_ylabel("Win Rate (%)")
ax.set_ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. 게임 시간 구간별 승률


In [ ]:
duration_bins = [0, 15, 20, 25, 30, 35, 40, float("inf")]
duration_labels = [
    "15분 이하", "15~20분", "20~25분", "25~30분",
    "30~35분", "35~40분", "40분 이상"
]

df["duration_group"] = pd.cut(
    df["game_duration_min"],
    bins=duration_bins,
    labels=duration_labels,
    include_lowest=True
)

duration_stats = (
    df.groupby("duration_group", observed=False)
    .agg(games=("win", "count"), wins=("win", "sum"), win_rate=("win", "mean"))
)

duration_stats["losses"] = duration_stats["games"] - duration_stats["wins"]
duration_stats["win_rate"] *= 100
duration_stats.round(2)


In [ ]:
ax = duration_stats["win_rate"].plot(kind="line", marker="o", figsize=(9, 5))
ax.set_title("Win Rate by Game Duration")
ax.set_xlabel("Game Duration")
ax.set_ylabel("Win Rate (%)")
ax.set_ylim(0, 100)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 9. 요일별 승률


In [ ]:
weekday_map = {0:"월", 1:"화", 2:"수", 3:"목", 4:"금", 5:"토", 6:"일"}
weekday_order = ["월", "화", "수", "목", "금", "토", "일"]

df["day_of_week"] = df["game_date"].dt.dayofweek.map(weekday_map)

weekday_stats = (
    df.groupby("day_of_week")
    .agg(games=("win", "count"), wins=("win", "sum"), win_rate=("win", "mean"))
    .reindex(weekday_order)
)

weekday_stats["losses"] = weekday_stats["games"] - weekday_stats["wins"]
weekday_stats["win_rate"] *= 100
weekday_stats.round(2)


In [ ]:
ax = weekday_stats["win_rate"].plot(kind="bar", figsize=(9, 5))
ax.set_title("Win Rate by Day of Week")
ax.set_xlabel("Day of Week")
ax.set_ylabel("Win Rate (%)")
ax.set_ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 10. 요일과 승패의 카이제곱 검정


In [ ]:
from scipy.stats import chi2_contingency

table = pd.crosstab(df["day_of_week"], df["win"]).reindex(weekday_order)
chi2, p, dof, expected = chi2_contingency(table)

print(f"Chi-square: {chi2:.3f}")
print(f"p-value: {p:.3f}")
print(f"Degrees of freedom: {dof}")

if p < 0.05:
    print("→ 요일과 승패 사이에 통계적으로 유의한 관계가 있습니다.")
else:
    print("→ 요일과 승패 사이에 통계적으로 유의한 관계가 없습니다.")


## 11. 챔피언별 성과


In [ ]:
champion_stats = (
    df.groupby("champion")
    .agg(
        games=("win", "count"),
        wins=("win", "sum"),
        win_rate=("win", "mean"),
        avg_kda=("kda", "mean"),
        avg_cs_per_min=("cs_per_min", "mean"),
        avg_damage_per_min=("damage_per_min", "mean")
    )
)

champion_stats["win_rate"] *= 100
champion_stats.sort_values(["games", "win_rate"], ascending=[False, False]).head(15).round(2)


In [ ]:
top_champions = champion_stats[champion_stats["games"] >= 5].sort_values("win_rate")

ax = top_champions["win_rate"].plot(kind="barh", figsize=(9, 6))
ax.set_title("Champion Win Rate (Minimum 5 Games)")
ax.set_xlabel("Win Rate (%)")
ax.set_ylabel("Champion")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()


## 12. 포지션별 성과


In [ ]:
position_stats = (
    df.groupby("position")
    .agg(
        games=("win", "count"),
        wins=("win", "sum"),
        win_rate=("win", "mean"),
        avg_kda=("kda", "mean")
    )
)

position_stats["win_rate"] *= 100
position_stats.sort_values("games", ascending=False).round(2)


## 13. 분석 정리

이 노트북에서는 개인 매치 데이터의 최종 경기 통계를 바탕으로 승패와 함께 나타나는 패턴을 확인했습니다.

- 승/패 경기의 주요 지표 차이
- 데스 횟수별 승률
- 게임 시간 구간별 승률
- 요일별 승률 및 통계적 유의성
- 챔피언/포지션별 성과

### 한계
최종 경기 통계는 경기 결과의 영향을 이미 포함하므로, 특정 지표가 승리를 **원인적으로 만들었다고 해석할 수 없습니다.**

### 향후 개선
Riot Timeline API를 활용해 10분/15분 시점의 지표를 수집하면 다음 분석이 가능합니다.

- 10분 골드 차이와 승률
- 15분 CS 차이와 승률
- 초반 데스 수와 최종 승패
- 오브젝트 관여와 승리 확률
- 초반 지표를 활용한 승패 예측 모델
